# Figure 3: Planning and Execution


In [ ]:
from pathlib import Path
import sys, json
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "experiments/intent_spec_example_mimic3_figure3.json").exists())
sys.path.insert(0, str(root / "experiments/figure3"))
from figure3 import load_spec, synthetic_database, run_figure3
spec = load_spec()
print(spec["question"])

## Synthetic execution


In [ ]:
db = synthetic_database(spec)
try:
    outcome = run_figure3(spec, db)
    assert outcome["status"] == "success", outcome
    assert outcome["execution_result"]["answer"] == 25.0, outcome
    print(outcome["query_plan"]["sql"])
    print(json.dumps(outcome["execution_result"], indent=2))
finally:
    db.close()

## Regression tests


In [ ]:
import unittest
from test_figure3 import Figure3Tests
result = unittest.TextTestRunner(verbosity=2).run(unittest.defaultTestLoader.loadTestsFromTestCase(Figure3Tests))
assert result.wasSuccessful()

## LLM planning on synthetic data


In [ ]:
RUN_LIVE_LLM = False
if RUN_LIVE_LLM:
    from llm_planner import run_live_demo
    try:
        live_outcome = run_live_demo()
    except Exception as exc:
        raise RuntimeError("Live setup/provider failed: " + type(exc).__name__) from None
    print(json.dumps(live_outcome, indent=2))
    assert live_outcome["validation_passed"], "Synthetic validation failed."
    (root / "experiments/figure3/live_run_result.json").write_text(json.dumps(live_outcome, indent=2) + "\n")


## Fixed IntentSpec on MIMIC-III


In [ ]:
RUN_REAL_DATABASE = False
if RUN_REAL_DATABASE:
    from run_real import run_real, DEFAULT_OUTPUT
    real_outcome = run_real()
    print("Status:", real_outcome["status"])
    print("Reference matches:", real_outcome["reference_check"]["matches"])
    print("Full result saved locally:", DEFAULT_OUTPUT)
    assert real_outcome["reference_check"]["matches"], "Reference mismatch."


## Figure 2 → Figure 3 integration


In [ ]:
RUN_INTEGRATED_WORKFLOW = True
if RUN_INTEGRATED_WORKFLOW:
    from workflow import run_integrated
    question = spec["question"]
    integrated_outcome = run_integrated(question)
    print("Workflow status:", integrated_outcome["status"])
    print("Upstream adjustments:", integrated_outcome["upstream_adjustments_used"])
    print("Full local handoff: experiments/figure3/local_results/figure4_integrated_handoff.json")
